In [ ]:
import torch
import torch.nn as nn

class ConcatBlockConv5(nn.Module):
    def __init__(self, in_ch, out_ch, k=32, act=nn.SiLU):
        super().__init__()
        
        def make_block(kernel_size):
            return nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding='same'),
                nn.BatchNorm1d(out_ch),
                act()
            )

        self.c1 = make_block(k)          # k=32
        self.c2 = make_block(k * 2)      # k=64
        self.c3 = make_block(k // 2)     # k=16
        self.c4 = make_block(k // 4)     # k=8
        self.c5 = make_block(k * 4)      # k=128
        
        c6_in_channels = (out_ch * 5) + in_ch
        
        self.c6 = nn.Sequential(
            nn.Conv1d(c6_in_channels, out_ch, kernel_size=1),
            nn.BatchNorm1d(out_ch),
            act()
        )

    def forward(self, x):
        x1 = self.c1(x)
        x2 = self.c2(x)
        x3 = self.c3(x)
        x4 = self.c4(x)
        x5 = self.c5(x)

        out = torch.cat([x1, x2, x3, x4, x5, x], dim=1)
        
        return self.c6(out)

In [ ]:
class GWNet(nn.Module):
    def __init__(self, channels=3, classes=1):
        super().__init__()

        self.b1 = ConcatBlockConv5(channels, 32, k=32)
        self.p1 = nn.MaxPool1d(2)

        self.b2 = ConcatBlockConv5(32, 64, k=32)
        self.p2 = nn.MaxPool1d(2)

        self.b3 = ConcatBlockConv5(64, 128, k=32)
        self.p3 = nn.MaxPool1d(2)

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(128, classes)

    def forward(self, x):
        x = self.p1(self.b1(x))
        x = self.p2(self.b2(x))
        x = self.p3(self.b3(x))
        x = self.gap(x).squeeze(-1)
        return self.fc(x)

In [ ]:
model = GWNet()
checkpoint_path = "residual_model_least_process_final.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=torch.device('cpu')))
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Model loaded successfully via state_dict!")

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error
from tqdm.notebook import tqdm  # Specialized progress bar for Jupyter
import os

# --- FILE PATHS ---
INPUT_FILE_LIST = "file.txt"
LABELS_CSV = "train_labels.csv"
OUTPUT_CSV = "grav_wave_analysis_results_full_tukey.csv"

# --- PHYSICS CONSTANTS ---
FS = 2048.0                       # Sampling Rate (Hz)
DURATION = 2.0                    # Duration (seconds)
N_POINTS = 4096                   # Number of data points
TIME_VECTOR = np.linspace(0, DURATION, N_POINTS)
DETECTORS = ['H1', 'L1', 'V1']    # Hanford, Livingston, Virgo

In [ ]:
def wave_model(t, A, omega, phase):
    return A * np.sin(omega * t + phase)
    
def load_and_preprocess(file_path):
    data = read_file(file_path)
    p1, p2, p3 = preprocess(data, 0)
    stacked = np.stack([p1, p2, p3], axis=0)
    return stacked

def get_model_probability(processed_data):
    input_tensor = torch.tensor(processed_data).float()
    input_tensor = input_tensor.unsqueeze(0)
    device = next(model.parameters()).device
    input_tensor = input_tensor.to(device)
    with torch.no_grad():
        outputs = model(input_tensor)
        prob = torch.sigmoid(outputs).item()
    return prob
import numpy as np
from scipy.optimize import curve_fit

def get_initial_guesses(t_segment, y_segment):
    # 1. Estimate Amplitude (Standard Deviation * sqrt(2) is a good proxy for Sine Amp)
    guess_A = np.std(y_segment) * np.sqrt(2)

    # 2. Estimate Frequency using FFT (Fast Fourier Transform)
    spectrum = np.fft.fft(y_segment)
    frequencies = np.fft.fftfreq(len(y_segment), d=(t_segment[1]-t_segment[0]))
    
    positive_freqs = frequencies[:len(frequencies)//2]
    positive_spectrum = np.abs(spectrum[:len(spectrum)//2])
    
    # Find the peak frequency
    peak_freq_index = np.argmax(positive_spectrum)
    peak_freq_hz = positive_freqs[peak_freq_index]
    
    # Convert to Angular Frequency (Omega)
    guess_omega = 2 * np.pi * peak_freq_hz
    
    # If FFT returns 0 (DC offset), force a fallback to LIGO's (100Hz)
    if guess_omega < 10.0: 
        guess_omega = 628.0 # Fallback to ~100 Hz

    return [guess_A, guess_omega, 0.0]

In [ ]:
df_labels = pd.read_csv("g2net-gravitational-wave-detection/training_labels.csv")

# Convert to dict: {'000a5b6': 1, '000a5b7': 0, ...}
true_labels_dict = pd.Series(df_labels.target.values, index=df_labels.id).to_dict()

print(f"Loaded {len(true_labels_dict)} labels.")

In [ ]:
with open("val_files_fullpaths.txt", 'r') as f:
    file_paths = [line.strip() for line in f if line.strip()]

print(f"Found {len(file_paths)} files to process.")

results = []


for file_path in tqdm(file_paths, desc="Processing Files"):
    try:
        # --- A. Extract File ID ---
        filename = file_path.split('/')[-1]   # Get last part
        file_id = filename.split('.')[0]      # Remove extension

        ground_truth = true_labels_dict.get(file_id, -1)

        data_3ch = load_and_preprocess(file_path)
        p_val = get_model_probability(data_3ch)

        for i in range(3):
            y_raw = data_3ch[i]
            
            # Initialize row with defaults (NaN)
            row = {
                'file_id': file_id,
                'detector': DETECTORS[i],
                'ground_truth': ground_truth, # Is it really a wave?
                'model_prob': p_val,          # Did your DL model think it's a wave?
                'fit_success': False,
                'A': np.nan,
                'omega': np.nan,
                'phase': np.nan,
                'snr': np.nan,
                'rmse': np.nan,
                'sigma_A': np.nan,
                'sigma_w': np.nan
            }
            try:

                p0 = get_initial_guesses(TIME_VECTOR, y_raw)

                params, pcov = curve_fit(wave_model, TIME_VECTOR, y_raw, p0=p0, 
                      bounds=([0, 0, -np.inf], [np.inf, np.inf, np.inf]), maxfev=5000)
                # Extract Parameters
                A_fit, w_fit, phi_fit = params
                # Extract Uncertainty
                perr = np.sqrt(np.diag(pcov)) 
                
                # Calculate Physics Metrics
                y_pred = wave_model(TIME_VECTOR, *params)
                residuals = y_raw - y_pred
                mse = mean_squared_error(y_raw, y_pred)
                rmse = np.sqrt(mse)
                noise_std = np.std(residuals)
                
                # Calculate SNR (Amplitude / Noise Floor)
                snr = abs(A_fit) / noise_std if noise_std > 1e-9 else 0.0
                # Save Valid Results
                row.update({
                    'fit_success': True,
                    'A': abs(A_fit),
                    'omega': w_fit,
                    'phase': phi_fit,
                    'snr': snr,
                    'rmse': rmse,
                    'sigma_A': perr[0],
                    'sigma_w': perr[1]
                })

            except (RuntimeError, ValueError):
                pass
            
            results.append(row)

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        continue

print("Processing Complete.")

In [ ]:
# Create DataFrame
df = pd.DataFrame(results)

cols = ['file_id', 'detector', 'ground_truth', 'model_prob', 'fit_success', 
        'A', 'omega', 'snr', 'rmse', 'sigma_A', 'sigma_w']
df = df[cols]

# Save to CSV
df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(df)} rows to {OUTPUT_CSV}")

# Display first few rows to verify
df.head()